# 策略对比: max(0,2,6) vs Extended vs max(2,6) 基准

比较两种策略相比 max(2,6) 基准的收益差异：
- **max(0,2,6)**: 取凌晨0点、2点、6点的最大值
- **Extended**: 取周五22:00到周六06:00每30分钟的最大值

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import warnings
warnings.filterwarnings('ignore')

# 中文字体设置
font_path = r'C:\Windows\Fonts\msyh.ttc'
chinese_font = FontProperties(fname=font_path, size=11)
chinese_font_title = FontProperties(fname=font_path, size=14)
chinese_font_legend = FontProperties(fname=font_path, size=10)

# 路径设置
OUTPUT_DIR = r'c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system\output'

# 交易量
USD_WEEKLY = 60_000_000  # 60M USD/周
JPY_WEEKLY = 4_000_000_000  # 4B JPY/周

print("配置完成！")

## 1. 加载数据

In [ ]:
# 加载数据
all_files = os.listdir(OUTPUT_DIR)
usdcnh_files = [f for f in all_files if 'USDCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
jpycnh_files = [f for f in all_files if 'JPYCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]

print(f"加载 USDCNH: {usdcnh_files[0]}")
df_usdcnh = pd.read_excel(os.path.join(OUTPUT_DIR, usdcnh_files[0]))
df_usdcnh['timestamp'] = pd.to_datetime(df_usdcnh['timestamp'])
df_usdcnh.set_index('timestamp', inplace=True)

print(f"加载 JPYCNH: {jpycnh_files[0]}")
df_jpycnh = pd.read_excel(os.path.join(OUTPUT_DIR, jpycnh_files[0]))
df_jpycnh['timestamp'] = pd.to_datetime(df_jpycnh['timestamp'])
df_jpycnh.set_index('timestamp', inplace=True)

# 添加日期列
for df in [df_usdcnh, df_jpycnh]:
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute
    df['weekday'] = df.index.dayofweek

print(f"\n数据范围: {df_usdcnh.index.min().date()} ~ {df_usdcnh.index.max().date()}")

## 2. 计算各策略的参照价和PnL

In [ ]:
# 获取周六和周五数据
sat_usdcnh = df_usdcnh[df_usdcnh['weekday'] == 5].copy()
fri_usdcnh = df_usdcnh[df_usdcnh['weekday'] == 4].copy()
sat_dates = sat_usdcnh['date'].unique()

# 构建每周数据
results = []
for sat_date in sat_dates:
    sat_day = sat_usdcnh[sat_usdcnh['date'] == sat_date]
    fri_date = (pd.Timestamp(sat_date) - pd.Timedelta(days=1)).date()
    fri_day = fri_usdcnh[fri_usdcnh['date'] == fri_date]
    
    # 获取 hour 0, 2, 6
    h0 = sat_day[(sat_day['hour'] == 0) & (sat_day['minute'] == 0)]
    h2 = sat_day[(sat_day['hour'] == 2) & (sat_day['minute'] == 0)]
    h6 = sat_day[(sat_day['hour'] == 6) & (sat_day['minute'] == 0)]
    
    if len(h0) == 0 or len(h2) == 0 or len(h6) == 0:
        continue
    
    usdcnh_h0 = h0['mid'].values[0]
    usdcnh_h2 = h2['mid'].values[0]
    usdcnh_h6 = h6['mid'].values[0]
    
    # Extended: 周五22:00 ~ 周六06:00 每30分钟
    extended_prices = []
    # 周五 22:00, 22:30, 23:00, 23:30
    for h in [22, 23]:
        for m in [0, 30]:
            fri_data = fri_day[(fri_day['hour'] == h) & (fri_day['minute'] == m)]
            if len(fri_data) > 0:
                extended_prices.append(fri_data['mid'].values[0])
    # 周六 00:00 ~ 06:00
    for h in range(7):
        for m in [0, 30]:
            if h == 6 and m == 30:
                continue
            sat_data = sat_day[(sat_day['hour'] == h) & (sat_day['minute'] == m)]
            if len(sat_data) > 0:
                extended_prices.append(sat_data['mid'].values[0])
    
    if len(extended_prices) < 10:
        continue
    
    # JPYCNH
    jpycnh_sat = df_jpycnh[df_jpycnh['date'] == sat_date]
    if len(jpycnh_sat) == 0:
        continue
    jpycnh_per_jpy = jpycnh_sat['mid'].mean()
    usdcnh_avg = sat_day['mid'].mean()
    
    # 参照价
    ref_max_26 = max(usdcnh_h2, usdcnh_h6)  # 基准
    ref_max_026 = max(usdcnh_h0, usdcnh_h2, usdcnh_h6)
    ref_extended = max(extended_prices)
    
    # JPYCNH 调整
    def calc_jpycnh(ref_usdcnh):
        return jpycnh_per_jpy * (ref_usdcnh / usdcnh_avg)
    
    jpycnh_max_26 = calc_jpycnh(ref_max_26)
    jpycnh_max_026 = calc_jpycnh(ref_max_026)
    jpycnh_extended = calc_jpycnh(ref_extended)
    
    # BPS 差异
    bps_026 = (ref_max_026 - ref_max_26) / ref_max_26 * 10000
    bps_ext = (ref_extended - ref_max_26) / ref_max_26 * 10000
    
    # PnL 差异
    usd_pnl_026 = (ref_max_026 - ref_max_26) * USD_WEEKLY
    usd_pnl_ext = (ref_extended - ref_max_26) * USD_WEEKLY
    jpy_pnl_026 = (jpycnh_max_026 - jpycnh_max_26) * JPY_WEEKLY
    jpy_pnl_ext = (jpycnh_extended - jpycnh_max_26) * JPY_WEEKLY
    
    results.append({
        'date': sat_date,
        'ref_max_26': ref_max_26,
        'ref_max_026': ref_max_026,
        'ref_extended': ref_extended,
        'bps_026': bps_026,
        'bps_ext': bps_ext,
        'usd_pnl_026': usd_pnl_026,
        'usd_pnl_ext': usd_pnl_ext,
        'jpy_pnl_026': jpy_pnl_026,
        'jpy_pnl_ext': jpy_pnl_ext,
        'total_pnl_026': usd_pnl_026 + jpy_pnl_026,
        'total_pnl_ext': usd_pnl_ext + jpy_pnl_ext,
    })

df = pd.DataFrame(results)
print(f"有效周数: {len(df)}")
print(f"回测期间: {df['date'].iloc[0]} ~ {df['date'].iloc[-1]}")

## 3. 每周平均 BPS 加价

In [ ]:
print("="*70)
print("每周平均 BPS 加价 vs max(2,6) 基准")
print("="*70)

print(f"\n{'策略':<40} {'平均 bps/周':>12} {'中位数':>10} {'最大':>10} {'最小':>10}")
print("-"*82)
print(f"{'max(0,2,6)':<40} {df['bps_026'].mean():>12.2f} {df['bps_026'].median():>10.2f} {df['bps_026'].max():>10.2f} {df['bps_026'].min():>10.2f}")
print(f"{'Extended (22:00~06:00 每30分钟)':<40} {df['bps_ext'].mean():>12.2f} {df['bps_ext'].median():>10.2f} {df['bps_ext'].max():>10.2f} {df['bps_ext'].min():>10.2f}")

print(f"\n📈 Extended 比 max(0,2,6) 每周平均多 {df['bps_ext'].mean() - df['bps_026'].mean():.2f} bps")

## 4. 累计 PnL (Accumulative PnL)

In [ ]:
# 计算累计 PnL
df['cum_usd_026'] = df['usd_pnl_026'].cumsum()
df['cum_jpy_026'] = df['jpy_pnl_026'].cumsum()
df['cum_total_026'] = df['total_pnl_026'].cumsum()

df['cum_usd_ext'] = df['usd_pnl_ext'].cumsum()
df['cum_jpy_ext'] = df['jpy_pnl_ext'].cumsum()
df['cum_total_ext'] = df['total_pnl_ext'].cumsum()

print("="*70)
print("累计 PnL vs max(2,6) 基准")
print("="*70)

print(f"\n{'策略':<40} {'USD (CNH)':>15} {'JPY (CNH)':>15} {'合计 (CNH)':>15}")
print("-"*85)
print(f"{'max(0,2,6)':<40} {df['cum_usd_026'].iloc[-1]:>15,.0f} {df['cum_jpy_026'].iloc[-1]:>15,.0f} {df['cum_total_026'].iloc[-1]:>15,.0f}")
print(f"{'Extended (22:00~06:00 每30分钟)':<40} {df['cum_usd_ext'].iloc[-1]:>15,.0f} {df['cum_jpy_ext'].iloc[-1]:>15,.0f} {df['cum_total_ext'].iloc[-1]:>15,.0f}")

diff = df['cum_total_ext'].iloc[-1] - df['cum_total_026'].iloc[-1]
print(f"\n📈 Extended 比 max(0,2,6) 累计多赚 {diff:,.0f} CNH ({diff/1e6:.2f}M)")

## 5. 累计 PnL 对比图

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 累计总 PnL 对比
ax1 = axes[0, 0]
ax1.plot(df['date'], df['cum_total_026']/1e6, 'b-', linewidth=2, marker='o', markersize=4, label='max(0,2,6)')
ax1.plot(df['date'], df['cum_total_ext']/1e6, 'r-', linewidth=2, marker='s', markersize=4, label='Extended')
ax1.set_title('累计 PnL 对比 (USD+JPY)', fontproperties=chinese_font_title)
ax1.set_xlabel('日期', fontproperties=chinese_font)
ax1.set_ylabel('累计 PnL (M CNH)', fontproperties=chinese_font)
ax1.legend(prop=chinese_font_legend)
ax1.grid(True, alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 添加最终值标注
ax1.annotate(f'{df["cum_total_026"].iloc[-1]/1e6:.1f}M', 
             xy=(df['date'].iloc[-1], df['cum_total_026'].iloc[-1]/1e6),
             xytext=(5, 0), textcoords='offset points', fontsize=10, color='blue')
ax1.annotate(f'{df["cum_total_ext"].iloc[-1]/1e6:.1f}M', 
             xy=(df['date'].iloc[-1], df['cum_total_ext'].iloc[-1]/1e6),
             xytext=(5, 0), textcoords='offset points', fontsize=10, color='red')

# 图2: USD 累计 PnL
ax2 = axes[0, 1]
ax2.plot(df['date'], df['cum_usd_026']/1e6, 'b-', linewidth=2, marker='o', markersize=4, label='max(0,2,6)')
ax2.plot(df['date'], df['cum_usd_ext']/1e6, 'r-', linewidth=2, marker='s', markersize=4, label='Extended')
ax2.set_title('USD 累计 PnL 对比', fontproperties=chinese_font_title)
ax2.set_xlabel('日期', fontproperties=chinese_font)
ax2.set_ylabel('累计 PnL (M CNH)', fontproperties=chinese_font)
ax2.legend(prop=chinese_font_legend)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 图3: JPY 累计 PnL
ax3 = axes[1, 0]
ax3.plot(df['date'], df['cum_jpy_026']/1e6, 'b-', linewidth=2, marker='o', markersize=4, label='max(0,2,6)')
ax3.plot(df['date'], df['cum_jpy_ext']/1e6, 'r-', linewidth=2, marker='s', markersize=4, label='Extended')
ax3.set_title('JPY 累计 PnL 对比', fontproperties=chinese_font_title)
ax3.set_xlabel('日期', fontproperties=chinese_font)
ax3.set_ylabel('累计 PnL (M CNH)', fontproperties=chinese_font)
ax3.legend(prop=chinese_font_legend)
ax3.grid(True, alpha=0.3)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 图4: 每周 BPS 对比
ax4 = axes[1, 1]
x = np.arange(len(df))
width = 0.35
bars1 = ax4.bar(x - width/2, df['bps_026'], width, label='max(0,2,6)', color='steelblue', alpha=0.8)
bars2 = ax4.bar(x + width/2, df['bps_ext'], width, label='Extended', color='indianred', alpha=0.8)
ax4.axhline(y=df['bps_026'].mean(), color='blue', linestyle='--', linewidth=1, label=f'max(0,2,6) 平均: {df["bps_026"].mean():.2f}')
ax4.axhline(y=df['bps_ext'].mean(), color='red', linestyle='--', linewidth=1, label=f'Extended 平均: {df["bps_ext"].mean():.2f}')
ax4.set_title('每周 BPS 加价对比', fontproperties=chinese_font_title)
ax4.set_xlabel('周', fontproperties=chinese_font)
ax4.set_ylabel('BPS vs max(2,6)', fontproperties=chinese_font)
ax4.legend(prop=chinese_font_legend, loc='upper right')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'strategy_comparison_final.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n图表已保存到: output/strategy_comparison_final.png")

## 6. 汇总统计

In [ ]:
print("="*70)
print("🏆 最终汇总 vs max(2,6) 基准")
print("="*70)

print(f"\n回测期间: {df['date'].iloc[0]} ~ {df['date'].iloc[-1]} ({len(df)} 周)")

print(f"\n┌{'─'*68}┐")
print(f"│{'策略':<30}│{'平均 bps/周':>18}│{'累计 PnL (M CNH)':>18}│")
print(f"├{'─'*30}┼{'─'*18}┼{'─'*18}┤")
print(f"│{'max(0,2,6)':<30}│{df['bps_026'].mean():>18.2f}│{df['cum_total_026'].iloc[-1]/1e6:>18.2f}│")
print(f"│{'Extended (22:00~06:00)':<30}│{df['bps_ext'].mean():>18.2f}│{df['cum_total_ext'].iloc[-1]/1e6:>18.2f}│")
print(f"└{'─'*30}┴{'─'*18}┴{'─'*18}┘")

bps_diff = df['bps_ext'].mean() - df['bps_026'].mean()
pnl_diff = (df['cum_total_ext'].iloc[-1] - df['cum_total_026'].iloc[-1]) / 1e6

print(f"\n📈 Extended 优势:")
print(f"   - 每周平均多加价: +{bps_diff:.2f} bps")
print(f"   - 累计多赚: +{pnl_diff:.2f}M CNH")
print(f"   - 年化多赚: +{pnl_diff * 52 / len(df):.2f}M CNH/年")

## 7. 分币种详情

In [ ]:
print("="*70)
print("分币种详情 vs max(2,6) 基准")
print("="*70)

print(f"\n【max(0,2,6) 策略】")
print(f"   USD: 累计 {df['cum_usd_026'].iloc[-1]:,.0f} CNH, 每周平均 {df['usd_pnl_026'].mean():,.0f} CNH")
print(f"   JPY: 累计 {df['cum_jpy_026'].iloc[-1]:,.0f} CNH, 每周平均 {df['jpy_pnl_026'].mean():,.0f} CNH")
print(f"   合计: 累计 {df['cum_total_026'].iloc[-1]:,.0f} CNH, 每周平均 {df['total_pnl_026'].mean():,.0f} CNH")

print(f"\n【Extended 策略 (22:00~06:00 每30分钟)】")
print(f"   USD: 累计 {df['cum_usd_ext'].iloc[-1]:,.0f} CNH, 每周平均 {df['usd_pnl_ext'].mean():,.0f} CNH")
print(f"   JPY: 累计 {df['cum_jpy_ext'].iloc[-1]:,.0f} CNH, 每周平均 {df['jpy_pnl_ext'].mean():,.0f} CNH")
print(f"   合计: 累计 {df['cum_total_ext'].iloc[-1]:,.0f} CNH, 每周平均 {df['total_pnl_ext'].mean():,.0f} CNH")

## 8. 每周明细数据

In [ ]:
display_df = df[['date', 'bps_026', 'bps_ext', 'total_pnl_026', 'total_pnl_ext', 'cum_total_026', 'cum_total_ext']].copy()
display_df.columns = ['日期', 'max(0,2,6) bps', 'Extended bps', 'max(0,2,6) PnL', 'Extended PnL', 'max(0,2,6) 累计', 'Extended 累计']

display(display_df.style.format({
    'max(0,2,6) bps': '{:.2f}',
    'Extended bps': '{:.2f}',
    'max(0,2,6) PnL': '{:,.0f}',
    'Extended PnL': '{:,.0f}',
    'max(0,2,6) 累计': '{:,.0f}',
    'Extended 累计': '{:,.0f}'
}))